# PC Build Advisor using OPT — Simplified Project Notebook

This notebook supports the GitHub/Streamlit project in the parent folder.

## What this notebook does
1. Load and clean the supplied PC component catalogue.
2. Audit every source attribute in the 11 component CSV files.
3. Extract user-stated component preferences, including common typing mistakes.
4. Use defaults from the PC recommendation engine for fields the user does not mention.
5. Test builds and the implemented compatibility rules.
6. Compare three requirement-extraction approaches:
   - Rule-based baseline
   - Original pretrained OPT-125M
   - Fine-tuned OPT-125M (LoRA)
7. Export comparison metrics and test results to `results/`.

**Important:** the Streamlit app uses `app.py`. This notebook is for data preparation, experiments, model comparison and reporting. It does not need to be run by Streamlit.

## 1. Setup

In [ ]:
from pathlib import Path
import sys, os, json, time, random, warnings
import numpy as np
import pandas as pd

# Works when the notebook is opened from the project root or /notebooks folder.
CANDIDATES = [Path.cwd(), Path.cwd().parent]
ROOT = next((p.resolve() for p in CANDIDATES if (p / "pc_engine.py").exists()), None)
if ROOT is None:
    matches = list(Path.cwd().glob("**/pc_engine.py"))
    ROOT = matches[0].parent.resolve() if matches else None
if ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Open this notebook inside the PCBuild-OPT project folder."
    )

sys.path.insert(0, str(ROOT))
CSV_DIR = ROOT / "csv"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Set RUN_MODEL_EVALUATION=True to compare all three methods.
# Set RETRAIN_EXTRACT_ADAPTER=True only if you intentionally want to retrain OPT-125M.
RUN_MODEL_EVALUATION = True
RETRAIN_EXTRACT_ADAPTER = False
pretrained_available = False
finetuned_available = False
pretrained_error = None
finetuned_error = None
TEST_SAMPLE_SIZE = 60
RANDOM_SEED = 42

print("Project root:", ROOT)
print("CSV folder exists:", CSV_DIR.exists())
print("GPU training/inference is available:", end=" ")
try:
    import torch
    print(torch.cuda.is_available())
except ImportError:
    print("PyTorch is not installed")

## 2. Import project modules

In [ ]:
import importlib
import pc_engine as E
import nlu_data as N
import catalog_nlu as C
import advisor as A

for module in (E, N, C, A):
    importlib.reload(module)

print("Project modules loaded.")
print("Requirement fields:", N.KEYS)
print("Available extraction modes:", A.available_modes())

## 3. Load and clean every component dataset

In [ ]:
data = E.load_data(CSV_DIR)
engine = E.BuildEngine(data)

cleaning_report = E.cleaning_report(CSV_DIR, data)
display(cleaning_report)

print("Cleaned product counts:")
display(pd.DataFrame([
    {"component": name, "usable_products": len(frame)}
    for name, frame in data.items()
]).sort_values("component"))

total_products = sum(len(frame) for frame in data.values())
print(f"Total usable component records: {total_products:,}")

## 4. Audit all dataset attributes

In [ ]:
# Every raw/derived field exposed to the requirement parser and review UI.
field_rows = []
for component, fields in C.FIELD_SCHEMA.items():
    frame = data.get(component)
    for field_name, label, data_column, kind in fields:
        available = (
            frame is not None
            and data_column in frame.columns
            and frame[data_column].notna().any()
        )
        field_rows.append({
            "component": component,
            "attribute": field_name,
            "description": label,
            "cleaned_column": data_column,
            "type": kind,
            "available_after_cleaning": bool(available),
        })

attribute_audit = pd.DataFrame(field_rows)
display(attribute_audit)
attribute_audit.to_csv(RESULTS_DIR / "attribute_audit.csv", index=False)

print("Total exposed attributes:", len(attribute_audit))
print("Unavailable after cleaning:",
      int((~attribute_audit["available_after_cleaning"]).sum()))

## 5. Inspect the cleaning and derived data

In [ ]:
# Sample a few useful derived columns to confirm the catalogue fields.
for component, columns in {
    "cpu": ["name", "brand", "socket", "cores", "boost", "tdp", "has_igpu"],
    "gpu": ["name", "brand", "chipset", "vram", "tdp"],
    "mobo": ["name", "socket", "form_factor", "chipset_code", "ddr"],
    "ram": ["name", "total_gb", "ddr", "mhz", "cas_latency"],
    "ssd": ["name", "capacity", "form_factor", "interface"],
    "psu": ["name", "watt", "efficiency", "modular"],
    "cooler": ["name", "size", "rpm_max", "noise_max"],
    "monitor": ["name", "screen_size", "resolution", "hz", "panel_type"],
    "keyboard": ["name", "style", "switches", "backlit", "tenkeyless", "connection_type"],
    "mouse": ["name", "tracking_method", "connection_type", "max_dpi", "hand_orientation"],
}.items():
    print("\\n", component.upper())
    available = [col for col in columns if col in data[component].columns]
    display(data[component][available].head(3))

## 6. Test natural-language extraction, typo correction and defaults

In [ ]:
sample_query = (
    "Bro, I need a wireles gaming mouse, black, 16000 DPI, "
    "a white ATX case with tempered glass, an IPS 27 inch 144Hz monitor, "
    "DDR5 6000 32GB RAM, 2TB NVMe SSD, 750W gold PSU, "
    "and RTX 5070 GPU. Budget RM15000."
)

sample_req = A.rule_requirements(sample_query, data)
print("Original:", sample_query)
print("Corrected:", sample_req["corrected_text"])
print("Typo corrections:", sample_req["typo_corrections"])
print("Core requirements:")
display(pd.DataFrame(
    [{"field": k, "value": sample_req.get(k)}
     for k in ["use_case", "budget", "currency", "cpu_brand", "gpu_brand",
               "ram_gb", "storage_gb", "extras", "compact", "budget_is_default"]]
))

sample_field_report = C.requirement_field_report(
    data, sample_req["component_preferences"], sample_req
)
display(sample_field_report[sample_field_report["Source"] == "User-stated"])

## 7. Generate a PC build using extracted component preferences

In [ ]:
sample_build, sample_violations = A.recommend(engine, sample_req)
if sample_build["ok"]:
    print(f"Build total: RM {sample_build['total_myr']:,.2f}")
    print(f"Budget: RM {sample_build['budget_myr']:,.2f}")
    print("Violations:", sample_violations)
    display(pd.DataFrame([
        {"component": item["category"], "product": item["name"],
         "price_myr": item["price_myr"], "specification": item["spec"]}
        for item in sample_build["items"]
    ]))
else:
    print("No compatible build:", sample_build["message"])

## 8. Prepare independent synthetic and OOD evaluation sets

In [ ]:
train_data = N.build_dataset(400, seed=RANDOM_SEED, mode="train")
test_data = N.build_dataset(TEST_SAMPLE_SIZE, seed=RANDOM_SEED + 1, mode="test")
ood_data = list(N.OOD_SET)

print("Train:", len(train_data))
print("Synthetic test:", len(test_data))
print("OOD test:", len(ood_data))
print("\\nSample test request:")
print(test_data[0][0])
print("Expected label:")
print(test_data[0][1])

## 9. Define one common evaluation helper

In [ ]:
def evaluate_one_method(method_name, dataset, predictions, seconds_per_example, split_name):
    labels = [label for _, label in dataset]
    metrics = N.evaluate(predictions, labels)
    return {
        "model": method_name,
        "test_set": split_name,
        "seconds_per_example": float(seconds_per_example),
        **metrics,
    }

def make_baseline_predictions(dataset):
    start = time.perf_counter()
    predictions = []
    for text, _ in dataset:
        corrected, _ = C.correct_typos(text)
        predictions.append(N.rule_extract(corrected))
    elapsed = (time.perf_counter() - start) / max(1, len(dataset))
    return predictions, elapsed

comparison_rows = []
prediction_cache = {}

## 10. Model 1 — Rule-based baseline

In [ ]:
for split_name, split_data in [("Synthetic", test_data), ("OOD", ood_data)]:
    preds, seconds = make_baseline_predictions(split_data)
    prediction_cache[("Rule-based baseline", split_name)] = preds
    comparison_rows.append(
        evaluate_one_method("Rule-based baseline", split_data, preds, seconds, split_name)
    )

print("Rule-based baseline evaluated.")

## 11. Model 2 — Original pretrained OPT-125M

In [ ]:
pretrained_available = False
pretrained_error = None

if RUN_MODEL_EVALUATION:
    try:
        import opt_nlu as O
        import torch
        pretrained_tokenizer, pretrained_model = O.load_opt(O.OPT_SIZES["125m"])

        for split_name, split_data in [("Synthetic", test_data), ("OOD", ood_data)]:
            texts = [C.correct_typos(text)[0] for text, _ in split_data]
            preds, seconds, raw = O.run_extraction(
                pretrained_model, pretrained_tokenizer,
                texts, shots=None, batch_size=4
            )
            prediction_cache[("Pretrained OPT-125M", split_name)] = preds
            comparison_rows.append(
                evaluate_one_method("Pretrained OPT-125M", split_data, preds, seconds, split_name)
            )

        pretrained_available = True
        del pretrained_model, pretrained_tokenizer
        O.free()
        print("Original OPT-125M evaluation completed.")
    except Exception as exc:
        pretrained_error = f"{type(exc).__name__}: {exc}"
        print("Original OPT-125M was not evaluated:", pretrained_error)
else:
    print("RUN_MODEL_EVALUATION=False; skipping original OPT download/inference.")

## 12. Model 3 — Fine-tuned OPT-125M (LoRA)

In [ ]:
finetuned_available = False
finetuned_error = None
adapter_dir = ROOT / "adapters" / "extract_125m"

if RUN_MODEL_EVALUATION:
    try:
        import opt_nlu as O

        if not (adapter_dir / "adapter_config.json").exists():
            if RETRAIN_EXTRACT_ADAPTER:
                pairs = O.extraction_pairs(train_data)
                train_seconds, train_loss = O.finetune_lora(
                    O.OPT_SIZES["125m"], pairs, str(adapter_dir),
                    epochs=2, lr=2e-4, batch_size=4,
                    grad_accum=4, max_len=640, grad_ckpt=True
                )
                print("Adapter trained:", train_seconds, "seconds; loss:", train_loss)
            else:
                raise FileNotFoundError(
                    "No adapters/extract_125m found. Set RETRAIN_EXTRACT_ADAPTER=True to train it."
                )

        finetuned_tokenizer, finetuned_model = O.load_finetuned(
            O.OPT_SIZES["125m"], str(adapter_dir)
        )

        for split_name, split_data in [("Synthetic", test_data), ("OOD", ood_data)]:
            texts = [C.correct_typos(text)[0] for text, _ in split_data]
            preds, seconds, raw = O.run_extraction(
                finetuned_model, finetuned_tokenizer,
                texts, shots=None, batch_size=4
            )
            prediction_cache[("Fine-tuned OPT-125M", split_name)] = preds
            comparison_rows.append(
                evaluate_one_method("Fine-tuned OPT-125M", split_data, preds, seconds, split_name)
            )

        finetuned_available = True
        del finetuned_model, finetuned_tokenizer
        O.free()
        print("Fine-tuned OPT-125M evaluation completed.")
    except Exception as exc:
        finetuned_error = f"{type(exc).__name__}: {exc}"
        print("Fine-tuned OPT-125M was not evaluated:", finetuned_error)
else:
    print("RUN_MODEL_EVALUATION=False; skipping fine-tuned OPT inference.")

## 13. Compare the three models\n\nThe comparison always contains Rule-based baseline, Pretrained OPT-125M, and Fine-tuned OPT-125M for both Synthetic and OOD tests. Models that could not be loaded are marked `Not evaluated`; no placeholder scores are invented. Run with `RUN_MODEL_EVALUATION=True` and the model dependencies available to obtain actual OPT scores.

In [ ]:
# Always show all three requested approaches, even if an OPT model could not be
# loaded in the current environment. Missing runs are labelled, never fabricated.
expected_models = [
    "Rule-based baseline",
    "Pretrained OPT-125M",
    "Fine-tuned OPT-125M",
]
expected_splits = ["Synthetic", "OOD"]
model_comparison = pd.DataFrame(comparison_rows)

if not model_comparison.empty:
    model_comparison["status"] = "Evaluated"
else:
    model_comparison = pd.DataFrame()

existing = set()
if not model_comparison.empty:
    existing = set(zip(model_comparison["model"], model_comparison["test_set"]))

placeholder_rows = []
metric_columns = [
    "json_valid_%", "exact_match_%", "mean_field_acc_%",
    "use_case_%", "budget_%", "cpu_brand_%", "gpu_brand_%",
    "ram_gb_%", "storage_gb_%", "seconds_per_example",
]
for model_name in expected_models:
    for split_name in expected_splits:
        if (model_name, split_name) not in existing:
            placeholder_rows.append({
                "model": model_name,
                "test_set": split_name,
                **{col: np.nan for col in metric_columns},
                "status": "Not evaluated",
            })

if placeholder_rows:
    model_comparison = pd.concat(
        [model_comparison, pd.DataFrame(placeholder_rows)],
        ignore_index=True,
    )

order = {name: i for i, name in enumerate(expected_models)}
model_comparison["_model_order"] = model_comparison["model"].map(order)
model_comparison["_split_order"] = model_comparison["test_set"].map(
    {"Synthetic": 0, "OOD": 1}
)
model_comparison = model_comparison.sort_values(
    ["_model_order", "_split_order"]
).drop(columns=["_model_order", "_split_order"])

display(model_comparison.round(2))
try:
    import matplotlib.pyplot as plt
    chart_data = model_comparison[
        model_comparison["status"] == "Evaluated"
    ].pivot(index="model", columns="test_set", values="exact_match_%")
    if not chart_data.empty:
        chart_data.plot(
            kind="bar", figsize=(9, 5), ylim=(0, 100),
            title="Exact-match accuracy: Rule-based vs original OPT vs fine-tuned OPT",
        )
        plt.ylabel("Exact-match accuracy (%)")
        plt.xlabel("Model / method")
        plt.xticks(rotation=15, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("No model has been evaluated yet. Run with RUN_MODEL_EVALUATION=True.")
except Exception as chart_error:
    print("Chart skipped:", chart_error)

model_comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
print("Saved three-model comparison to:", RESULTS_DIR / "model_comparison.csv")


## 14. Inspect incorrect predictions

In [ ]:
# Show a few exact-match errors for each evaluated model on the OOD set.
gold_ood = [label for _, label in ood_data]
for model_name in ["Rule-based baseline", "Pretrained OPT-125M", "Fine-tuned OPT-125M"]:
    preds = prediction_cache.get((model_name, "OOD"))
    if preds is None:
        continue
    errors = []
    for (text, gold), pred in zip(ood_data, preds):
        normal_pred = N.normalise(pred) if pred is not None else None
        if normal_pred != N.normalise(gold):
            errors.append({
                "request": text,
                "expected": N.to_json(gold),
                "predicted": N.to_json(normal_pred) if normal_pred else "Invalid JSON",
            })
    print("\\n", model_name, "- errors:", len(errors), "/", len(ood_data))
    if errors:
        display(pd.DataFrame(errors).head(5))

## 15. Test that unmentioned attributes use defaults

In [ ]:
queries = [
    "I need a gaming PC around RM5000.",
    "Build a programming PC with an Intel CPU, budget RM6000.",
    "Need a streaming computer, budget RM8000 and 32GB RAM.",
]

for text in queries:
    req = A.rule_requirements(text, data)
    report = C.requirement_field_report(data, req["component_preferences"], req)
    print("\\nREQUEST:", text)
    print("Budget:", req["budget"], "default used:", req["budget_is_default"])
    print("Explicit dataset attributes:", req["attribute_count"])
    print("Unmentioned attributes:", int((report["Source"] == "Default").sum()))

## 16. Budget strategy regression test

In [ ]:
budget_rows = []
for budget in [5000, 10000]:
    req = A.rule_requirements(
        f"Gaming PC, budget RM{budget}", data
    )
    req["budget_strategy"] = "closest_to_budget"
    result, violations = A.recommend(engine, req)
    budget_rows.append({
        "budget_myr": budget,
        "build_ok": result["ok"],
        "total_myr": result.get("total_myr"),
        "remaining_myr": (
            budget - result["total_myr"] if result["ok"] else None
        ),
        "violations": len(violations),
    })

budget_test = pd.DataFrame(budget_rows)
display(budget_test)

## 17. Export all project experiment results

In [ ]:
cleaning_report.to_csv(RESULTS_DIR / "cleaning_report.csv", index=False)
attribute_audit.to_csv(RESULTS_DIR / "attribute_audit.csv", index=False)
budget_test.to_csv(RESULTS_DIR / "budget_strategy_test.csv", index=False)

summary = {
    "usable_product_records": int(total_products),
    "attributes_audited": int(len(attribute_audit)),
    "test_examples": int(len(test_data)),
    "ood_examples": int(len(ood_data)),
    "pretrained_evaluated": bool(pretrained_available),
    "fine_tuned_evaluated": bool(finetuned_available),
    "pretrained_error": pretrained_error,
    "fine_tuned_error": finetuned_error,
}
(RESULTS_DIR / "notebook_run_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
run_mode_lines = [
    f"RUN_MODEL_EVALUATION={RUN_MODEL_EVALUATION}",
    f"RETRAIN_EXTRACT_ADAPTER={RETRAIN_EXTRACT_ADAPTER}",
    f"Pretrained OPT evaluated={pretrained_available}",
    f"Fine-tuned OPT evaluated={finetuned_available}",
    f"Pretrained error={pretrained_error}",
    f"Fine-tuned error={finetuned_error}",
]
(RESULTS_DIR / "run_mode.txt").write_text(
    "\n".join(run_mode_lines) + "\n", encoding="utf-8"
)
print(json.dumps(summary, indent=2))
print("\nSaved results to:", RESULTS_DIR)


## 18. Interpretation notes for the assignment report

- **Rule-based baseline** is deterministic and does not download a Transformer.
- **Pretrained OPT-125M** is the original Hugging Face checkpoint with no project fine-tuning.
- **Fine-tuned OPT-125M** uses the project's `adapters/extract_125m` LoRA adapter.
- All three are evaluated on the same synthetic and OOD test examples.
- Compare JSON validity, exact match, mean field accuracy and seconds per example.
- Report missing/failed model evaluation honestly; do not invent scores.
- A lower exact-match score on the OOD set indicates weaker generalization to unseen phrasings.
- The PC compatibility engine is a separate rule-based component. Model performance measures requirement extraction, not hardware compatibility itself.

**Reproducibility:** keep the random seed, test split, adapter version and package versions with the submitted report.